In [ ]:
from ultralytics import YOLO, SAM
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import supervision as sv
from pathlib import Path
import os

# Cargar imagen de referencia
image_bgr = cv2.imread("assets/futbot-01.jpg")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Recalcular H (mismos puntos que NB10 — notebook standalone)
SOURCE_POINTS = np.float32([[20, 165], [470, 110], [560, 920], [15, 985]])
CAMPO_W, CAMPO_H = 364, 486
ESCALA_PX_CM = 2.0  # 1 cm real = 2 px en el campo canónico (RCJ SF23: 182 × 243 cm)
TARGET_POINTS = np.float32([[0, 0], [CAMPO_W, 0], [CAMPO_W, CAMPO_H], [0, CAMPO_H]])
H = cv2.getPerspectiveTransform(SOURCE_POINTS, TARGET_POINTS)

CLASS_NAMES = {0: "azul", 1: "rojo", 2: "balón"}
COLORS_HEX  = {0: "#00b4d8", 1: "#ef233c", 2: "#ff9500"}
COLORS_BGR  = {0: (216, 180, 0), 1: (60, 35, 239), 2: (0, 149, 255)}


def detect_robots_hsv(frame_bgr: np.ndarray,
                      min_area: int = 20) -> sv.Detections:
    """
    Detecta robots y balón en un frame usando filtros de color HSV.
    Devuelve sv.Detections con class_id: 0=azul, 1=rojo, 2=balón.
    """
    #1. blur slight
    blurred = cv2.GaussianBlur(frame_bgr, (5,5),0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

    mask_azul = cv2.inRange(hsv, np.array([85,  100,  180]),
                                 np.array([95, 255, 255]))
    
    mask_rojo = cv2.inRange(hsv, np.array([  0, 100,  100]),
                                 np.array([ 10, 255, 255]))
    
    mask_rojo |= cv2.inRange(hsv, np.array([170, 100,  100]),
                                  np.array([10, 255, 255]))
    
    mask_balon = cv2.inRange(hsv, np.array([  5, 150, 150]),
                                  np.array([ 20, 255, 255]))

    xyxy_list, class_ids = [], []
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    
    for mask, cid in [(mask_azul, 0), (mask_rojo, 1), (mask_balon, 2)]:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue
            x, y, w, h_box = cv2.boundingRect(cnt)
            pad_w = 40
            pad_h = 40

            x1 = max(0, x - pad_w)
            y1 = max(0, y - pad_h)
            x2 = min(frame_bgr.shape[1], x + w + pad_w)
            y2 = min(frame_bgr.shape[0], y + h_box + pad_h)

            xyxy_list.append([x1, y1, x2, y2])
            class_ids.append(cid)


    if not xyxy_list:
        return sv.Detections.empty()

    return sv.Detections(
        xyxy=np.array(xyxy_list, dtype=np.float32),
        class_id=np.array(class_ids, dtype=int),
    )
dets_hsv = detect_robots_hsv(image_bgr)
print(f"Detecciones: HSV: {len(dets_hsv)} - bboxes que usaremos como prompt para SAM")

# Probar en la imagen estática
dets = detect_robots_hsv(image_bgr)
print(f"Detecciones encontradas: {len(dets)}")
for i, (box, cid) in enumerate(zip(dets.xyxy, dets.class_id)):
    cx = int((box[0] + box[2]) / 2)
    cy = int(box[3])
    print(f"  [{i}] {CLASS_NAMES[cid]:5s}: bbox={box.astype(int).tolist()}  bottom_center=({cx},{cy})")


# Visualizar detecciones sobre la imagen
#vis = image_rgb.copy()
#for box, cid in zip(dets.xyxy, dets.class_id):
#    x1, y1, x2, y2 = box.astype(int)
#    color_bgr = COLORS_BGR[cid]
#    color_rgb = (color_bgr[2], color_bgr[1], color_bgr[0])
#    cv2.rectangle(vis, (x1, y1), (x2, y2), color_rgb, 3)
#    cx, cy = (x1 + x2) // 2, y2
#    cv2.circle(vis, (cx, cy), 8, color_rgb, -1)  # BOTTOM_CENTER
#    cv2.putText(vis, CLASS_NAMES[cid], (x1, y1 - 6),
#                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_rgb, 2)

#plt.figure(figsize=(5.6, 10))
#plt.imshow(vis)
#plt.title(f"detect_robots_hsv: {len(dets)} detecciones (● = BOTTOM_CENTER)")
#plt.axis("off")
#plt.tight_layout()
#plt.show()

sam_model = SAM("sam3.pt")
print("SAM CARGADO")

bboxes = dets_hsv.xyxy.tolist()

if not bboxes:
    dets_sam = sv.Detections.empty()
else:
    results = sam_model(image_bgr, bboxes = bboxes, verbose = False)
    dets_sam = sv.Detections.from_ultralytics(results[0])

if len(dets_sam) == len(dets_hsv):
    dets_sam.class_id = dets_hsv.class_id

print(f"Detecciones SAM {len(dets_sam)}")
print(f"Mascaras shape: {dets_sam.mask.shape if dets_sam.mask is not None else 'None'}")
print(f"Dtype: {dets_sam.mask.dtype if dets_sam.mask is not None else 'None'}")

palette = sv.ColorPalette.from_hex(list(COLORS_HEX.values()))
mask_ann = sv.MaskAnnotator(color=palette, opacity = 0.5)
box_ann = sv.BoxAnnotator(color=palette, thickness = 2)
label_ann = sv.LabelAnnotator(color=palette, text_color = sv.Color.WHITE)

labels = [CLASS_NAMES.get(int(c), "?") for c in (dets_sam.class_id if dets_sam.class_id is not None else [])]
vis = mask_ann.annotate(image_rgb.copy(), dets_sam)
vis = box_ann.annotate(vis, dets_sam)
vis = label_ann.annotate(vis, dets_sam, labels=labels)

plt.figure(figsize=(5.6,10))
plt.imshow(vis)
plt.title(f"SAM: {len(dets_sam)} mascara de robots (opacity = 0.5)")
plt.axis("off")
plt.tight_layout()
plt.show()

def project_mask_contour(mask: np.ndarray, H: np.ndarray) -> np.ndarray | None:
    """Proyecta el contorno exterior de una máscara al campo canónico."""
    contours, _ = cv2.findContours(mask.astype(np.uint8),
                                   cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 1, 2).astype(np.float32)
    pts_proj = cv2.perspectiveTransform(pts, H)
    return pts_proj.reshape(-1, 2).astype(np.int32)


def draw_tactical_with_masks(dets_sam: sv.Detections, H: np.ndarray,
                              campo_w: int = CAMPO_W, campo_h: int = CAMPO_H) -> np.ndarray:
    """Dibuja el mapa táctico con rellenos y contornos de máscaras SAM proyectadas."""
    canvas = np.zeros((campo_h, campo_w, 3), dtype=np.uint8)
    canvas[:] = (50, 67, 27)

    # Líneas del campo
    cv2.rectangle(canvas, (0, 0), (campo_w - 1, campo_h - 1), (120, 200, 116), 2)
    cv2.line(canvas, (0, campo_h // 2), (campo_w, campo_h // 2), (120, 200, 116), 1)
    cv2.circle(canvas, (campo_w // 2, campo_h // 2), int(30 * ESCALA_PX_CM), (120, 200, 116), 1)
    pen_w = int(80 * ESCALA_PX_CM); pen_h = int(40 * ESCALA_PX_CM)
    pen_x = (campo_w - pen_w) // 2
    cv2.rectangle(canvas, (pen_x, 0), (pen_x + pen_w, pen_h), (120, 200, 116), 1)
    cv2.rectangle(canvas, (pen_x, campo_h - pen_h),
                           (pen_x + pen_w, campo_h - 1), (120, 200, 116), 1)
    goal_wp = int(60 * ESCALA_PX_CM); goal_xp = (campo_w - goal_wp) // 2
    cv2.rectangle(canvas, (goal_xp, 0), (goal_xp + goal_wp, int(8 * ESCALA_PX_CM)), (0, 214, 255), 2)
    cv2.rectangle(canvas, (goal_xp, campo_h - int(8 * ESCALA_PX_CM)),
                           (goal_xp + goal_wp, campo_h - 1), (216, 180, 0), 2)

    masks     = dets_sam.mask
    class_ids = (dets_sam.class_id if dets_sam.class_id is not None
                 else np.zeros(len(dets_sam), dtype=int))

    if masks is not None:
        for m, cid in zip(masks, class_ids):
            color = COLORS_BGR.get(int(cid), (200, 200, 200))

             #Opción A: relleno semitransparente (warp de la máscara)
            m_uint8  = m.astype(np.uint8) * 255
            m_warped = cv2.warpPerspective(m_uint8, H, (campo_w, campo_h))
            overlay  = canvas.copy()
            overlay[m_warped > 0] = color
            canvas = cv2.addWeighted(overlay, 0.45, canvas, 0.55, 0)

            # Opción B: contorno proyectado
            #contorno = project_mask_contour(m, H)
            #if contorno is not None and len(contorno) > 2:
             #   cv2.polylines(canvas, [contorno], True, color, 2)

            # Centroide de la máscara
            ys, xs = np.where(m)
            if len(xs) > 0:
                cx = int(xs.mean()); cy = int(ys.mean())
                pt = np.float32([[[cx, cy]]])
                proj = cv2.perspectiveTransform(pt, H)
                px, py = int(proj[0][0][0]), int(proj[0][0][1])
                if 0 <= px < campo_w and 0 <= py < campo_h:
                    cv2.circle(canvas, (px, py), 8, color, -1)
                    cv2.circle(canvas, (px, py), 8, (255, 255, 255), 1)

    return canvas


tactical_bgr = draw_tactical_with_masks(dets_sam, H)
plt.figure(figsize=(6, 8.0))
plt.imshow(cv2.cvtColor(tactical_bgr, cv2.COLOR_BGR2RGB))
plt.title("Mapa táctico con máscaras SAM proyectadas — relleno + contorno")
plt.axis("off")
plt.tight_layout()
plt.show()

#mapa de calor

def build_heatmap(video_path: str, sam_model, H: np.ndarray,
                  campo_w: int = CAMPO_W, campo_h: int = CAMPO_H,
                  max_frames: int = 24) -> dict:
    """
    Procesa los primeros max_frames del video y acumula la presencia
    de cada equipo en el campo canónico.
    Devuelve {class_id: ndarray (campo_h, campo_w) acumulado}.
    """
    heatmaps = {0: np.zeros((campo_h, campo_w), dtype=np.float32),
                1: np.zeros((campo_h, campo_w), dtype=np.float32)}
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret: break
        dets = detect_robots_hsv(frame)
        if len(dets) > 0:
            results  = sam_model(frame, bboxes=dets.xyxy.tolist(), verbose=False)
            dets_sam = sv.Detections.from_ultralytics(results[0])
            if len(dets_sam) == len(dets):
                dets_sam.class_id = dets.class_id
            if dets_sam.mask is not None:
                ids_validos = dets_sam.class_id if dets_sam.class_id is not None else[]
                for m, cid in zip(dets_sam.mask, ids_validos):
                    if int(cid) in heatmaps:
                        m_w = cv2.warpPerspective(m.astype(np.uint8) * 255, H,
                                                  (campo_w, campo_h))
                        heatmaps[int(cid)] += m_w.astype(np.float32)
        frame_count += 1
    cap.release()
    # Normalizar a [0, 1]
    for cid in heatmaps:
        mx = heatmaps[cid].max()
        if mx > 0:
            heatmaps[cid] /= mx
    return heatmaps

print("Generando mapas de calor (primeros 24 frames)...")
heatmaps = build_heatmap("assets/futbot-2s.mp4", sam_model, H)

fig, axes = plt.subplots(1, 2, figsize=(8, 6))
for ax, (cid, hm) in zip(axes, heatmaps.items()):
    warped_rgb = cv2.cvtColor(
        cv2.warpPerspective(image_bgr, H, (CAMPO_W, CAMPO_H)),
        cv2.COLOR_BGR2RGB)
    ax.imshow(warped_rgb, alpha=0.4)
    cmap = "Blues" if cid == 0 else "Reds"
    ax.imshow(hm, alpha=0.6, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(f"Equipo {CLASS_NAMES[cid]} — mapa de calor (24 frames)")
    ax.axis("off")
plt.suptitle("Control del campo a lo largo del video", fontsize=13)
plt.tight_layout()
plt.show()



#codificacion de video

OUTPUT_VIDEO = "assets/futbot1.mp4"
SOURCE_VIDEO = "assets/futbot-2s.mp4"

os.makedirs(os.path.dirname(OUTPUT_VIDEO), exist_ok=True)

def callback_nb12(frame: np.ndarray, _:int) -> np.ndarray:
    escala = CAMPO_H / frame.shape[0]
    orig_w = int(frame.shape[1] * escala)

    if orig_w % 2 != 0:
        orig_w +=1
    
    dets = detect_robots_hsv(frame)
    if len(dets) == 0:
        resized = cv2.resize(frame, (orig_w, CAMPO_H))
        blank = np.zeros((CAMPO_H, CAMPO_W, 3), dtype=np.uint8); blank[:] = (50, 67, 27)
        return np.hstack([resized, blank, blank])
    
    results = sam_model(frame, bboxes=dets.xyxy.tolist(), verbose=False)
    dets_sam = sv.Detections.from_ultralytics(results[0])
    if len(dets_sam) == len(dets):
        dets_sam.class_id = dets.class_id
    
    warped = cv2.warpPerspective(frame,H, (CAMPO_W, CAMPO_H))

    tactical = draw_tactical_with_masks(dets_sam, H)

    resized = cv2.resize(frame, (orig_w, CAMPO_H))
    if dets_sam.mask is not None:
        scale_x = orig_w / frame.shape[1]
        scale_y = CAMPO_H / frame.shape[0]

        scaled_xyxy = dets_sam.xyxy.copy()
        scaled_xyxy[:, [0, 2]] = scaled_xyxy[:, [0, 2]] * scale_x
        scaled_xyxy[:, [1, 3]] = scaled_xyxy[:, [1, 3]] * scale_y

        scaled_mask = []
        for mask_item in dets_sam.mask:
            resized_mask_item = cv2.resize(mask_item.astype(np.uint8), (orig_w, CAMPO_H), interpolation = cv2.INTER_NEAREST)
            scaled_mask.append(resized_mask_item.astype(bool))
            scaled_masks = np.array(scaled_mask)

        dets_sam_for_annottation = sv.Detections(
            xyxy = scaled_xyxy,
            mask = scaled_masks,
            confidence = dets_sam.confidence,
            class_id = dets_sam.class_id
        )

        resized_rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        resized_rgb = sv.MaskAnnotator(opacity = 0.4).annotate(resized_rgb, dets_sam_for_annottation)
        resized = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2BGR)

    return np.hstack([resized, warped, tactical])

print("Iniciando el proceso")

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
generator = sv.get_video_frames_generator(SOURCE_VIDEO)

first_frame = next(sv.get_video_frames_generator(SOURCE_VIDEO))
test_output = callback_nb12(first_frame, 0)
new_height, new_width = test_output.shape[:2]

custom_video_info = sv.VideoInfo(
    width = new_width,
    height = new_height,
    fps=video_info.fps,
    total_frames=video_info.total_frames
)

print(f"original resolucion: {video_info.width} x {video_info.height}")
print(f"Nueva resolucion (hstack): {new_width} x {new_height}")

with sv.VideoSink(target_path=OUTPUT_VIDEO, video_info=custom_video_info) as sink:
    for i, frame in enumerate(generator):
        result_frame = callback_nb12(frame, i)
        sink.write_frame(frame=result_frame)
        if i % 10 == 0:
            print(f"Procesando frame {i} de {video_info.total_frames}...")

print(f"video raw guardado con exito: {OUTPUT_VIDEO}")